In [5]:
import os
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

# YOLO klasör dizinlerinin hazırlanması
BASE_DIR = "../dataset/yolo"
for split in ['train', 'val']:
    os.makedirs(os.path.join(BASE_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(BASE_DIR, 'labels', split), exist_ok=True)

print("FLIR veri seti yükleniyor...")
dataset = load_dataset("Francesco/flir-camera-objects")

# Sınıf isimlerini veri yapısına göre güvenli şekilde alma
cat_names = None
feat_objects = dataset['train'].features['objects']

if isinstance(feat_objects, list) and len(feat_objects) > 0 and 'category' in feat_objects[0]:
    cat_feat = feat_objects[0]['category']
    if hasattr(cat_feat, 'names'):
        cat_names = cat_feat.names
elif hasattr(feat_objects, 'feature') and 'category' in feat_objects.feature:
    cat_feat = feat_objects.feature['category']
    if hasattr(cat_feat, 'names'):
        cat_names = cat_feat.names
elif isinstance(feat_objects, dict) and 'category' in feat_objects:
    cat_feat = feat_objects['category']
    if hasattr(cat_feat, 'names'):
        cat_names = cat_feat.names

print("Veri setindeki sınıflar:", cat_names)

# car -> 0, person -> 1 haritalaması
target_map = {}
if cat_names:
    for idx, name in enumerate(cat_names):
        name_lower = str(name).lower()
        if 'car' in name_lower or 'vehicle' in name_lower:
            target_map[idx] = 0
        elif 'person' in name_lower or 'pedestrian' in name_lower or 'people' in name_lower:
            target_map[idx] = 1
else:
    target_map = {0: 0, 1: 1}

print(f"Hedef Haritalaması (Orijinal ID -> YOLO ID): {target_map}")

def convert_to_yolo(hf_split_name, yolo_split_name):
    print(f"\n{yolo_split_name.upper()} verileri dönüştürülüyor...")
    saved_count = 0
    
    for idx, item in enumerate(tqdm(dataset[hf_split_name])):
        image = item['image']
        width, height = image.size
        annotations = item['objects']
        
        # Format uyuşmazlığına karşı liste / sözlük kontrolü
        if isinstance(annotations, list):
            bboxes = [obj['bbox'] for obj in annotations if 'bbox' in obj]
            categories = [obj['category'] for obj in annotations if 'category' in obj]
        else:
            bboxes = annotations['bbox']
            categories = annotations['category']
            
        yolo_lines = []
        for box, cat_id in zip(bboxes, categories):
            if cat_id in target_map:
                yolo_cls = target_map[cat_id]
                x_min, y_min, w, h = box
                
                # YOLO formatı koordinatları (x_center, y_center, width, height)
                x_center = (x_min + w / 2.0) / width
                y_center = (y_min + h / 2.0) / height
                norm_w = w / width
                norm_h = h / height
                
                # Koordinat taşmalarını 0.0 - 1.0 sınırlarına sabitleme
                x_center = max(0.0, min(1.0, x_center))
                y_center = max(0.0, min(1.0, y_center))
                norm_w = max(0.0, min(1.0, norm_w))
                norm_h = max(0.0, min(1.0, norm_h))
                
                yolo_lines.append(f"{yolo_cls} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")
        
        # Yalnızca hedef nesne içeren tam kareleri kaydet
        if yolo_lines:
            img_filename = f"flir_{yolo_split_name}_{idx:05d}.jpg"
            lbl_filename = f"flir_{yolo_split_name}_{idx:05d}.txt"
            
            img_path = os.path.join(BASE_DIR, 'images', yolo_split_name, img_filename)
            lbl_path = os.path.join(BASE_DIR, 'labels', yolo_split_name, lbl_filename)
            
            image.save(img_path, quality=95)
            with open(lbl_path, 'w') as f:
                f.write("\n".join(yolo_lines))
            saved_count += 1
            
    print(f"{yolo_split_name.upper()} tamamlandı: {saved_count} görsel ve etiket oluşturuldu.")

# Split isimlerini otomatik belirleyip dönüştürmeyi başlatma
val_key = 'validation' if 'validation' in dataset else 'val'
convert_to_yolo('train', 'train')
convert_to_yolo(val_key, 'val')

FLIR veri seti yükleniyor...
Veri setindeki sınıflar: None
Hedef Haritalaması (Orijinal ID -> YOLO ID): {0: 0, 1: 1}

TRAIN verileri dönüştürülüyor...


100%|██████████| 9306/9306 [00:23<00:00, 394.94it/s]


TRAIN tamamlandı: 1246 görsel ve etiket oluşturuldu.

VAL verileri dönüştürülüyor...


100%|██████████| 1452/1452 [00:04<00:00, 317.85it/s]

VAL tamamlandı: 276 görsel ve etiket oluşturuldu.
